<a href="https://colab.research.google.com/github/lekhasree15/BIS_LAB/blob/main/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# ============================================================
# GENETIC ALGORITHM FOR INVESTMENT PORTFOLIO OPTIMIZATION
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE THE PROBLEM
# ------------------------------------------------------------

# Four example stocks
stocks = [
    "Stock A",
    "Stock B",
    "Stock C",
    "Stock D"
]

# Expected annual returns of the stocks
expected_returns = np.array([
    0.12,   # Stock A = 12%
    0.10,   # Stock B = 10%
    0.15,   # Stock C = 15%
    0.08    # Stock D = 8%
])

# Standard deviations (risk) of the stocks
std_deviation = np.array([
    0.20,
    0.15,
    0.25,
    0.10
])

# Correlation/covariance information
correlation_matrix = np.array([
    [1.00, 0.20, 0.30, 0.10],
    [0.20, 1.00, 0.15, 0.25],
    [0.30, 0.15, 1.00, 0.05],
    [0.10, 0.25, 0.05, 1.00]
])

# Covariance matrix
covariance_matrix = (
    correlation_matrix *
    np.outer(std_deviation, std_deviation)
)


# ------------------------------------------------------------
# 2. INITIALIZE PARAMETERS
# ------------------------------------------------------------

population_size = 10
crossover_rate = 0.60
mutation_rate = 0.40
number_of_generations = 100

number_of_stocks = len(stocks)


# ------------------------------------------------------------
# 3. CREATE INITIAL POPULATION
# ------------------------------------------------------------

def create_population():
    """
    Creates random portfolios.

    Each chromosome represents one portfolio.
    Each gene represents the weight of one stock.
    """

    population = []

    for _ in range(population_size):

        # Generate random positive values
        chromosome = np.random.random(number_of_stocks)

        # Normalize so that total investment = 100%
        chromosome = chromosome / np.sum(chromosome)

        population.append(chromosome)

    return np.array(population)


# ------------------------------------------------------------
# 4. EVALUATE FITNESS
# ------------------------------------------------------------

def calculate_return(weights):
    """
    Portfolio expected return.
    """

    return np.dot(weights, expected_returns)


def calculate_risk(weights):
    """
    Portfolio risk calculated using covariance matrix.
    """

    variance = np.dot(
        weights.T,
        np.dot(covariance_matrix, weights)
    )

    return np.sqrt(variance)


def fitness_function(weights):
    """
    Fitness = Return / Risk

    Higher fitness is better.
    """

    portfolio_return = calculate_return(weights)
    portfolio_risk = calculate_risk(weights)

    return portfolio_return / portfolio_risk


# ------------------------------------------------------------
# 5. SELECTION
# ------------------------------------------------------------

def roulette_wheel_selection(population, fitness_values):
    """
    Roulette-wheel selection.

    Individuals with higher fitness have
    a higher probability of being selected.
    """

    # Make all fitness values positive
    fitness_values = fitness_values - np.min(fitness_values) + 0.000001

    probabilities = fitness_values / np.sum(fitness_values)

    selected_index = np.random.choice(
        len(population),
        p=probabilities
    )

    return population[selected_index].copy()


# ------------------------------------------------------------
# 6. CROSSOVER
# ------------------------------------------------------------

def arithmetic_crossover(parent1, parent2):
    """
    Arithmetic crossover.

    Child is created using a weighted combination
    of the two parents.
    """

    if np.random.random() < crossover_rate:

        alpha = np.random.random()

        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = alpha * parent2 + (1 - alpha) * parent1

    else:

        child1 = parent1.copy()
        child2 = parent2.copy()

    # Normalize the weights
    child1 = child1 / np.sum(child1)
    child2 = child2 / np.sum(child2)

    return child1, child2


# ------------------------------------------------------------
# 7. MUTATION
# ------------------------------------------------------------

def mutation(chromosome):
    """
    Mutation randomly changes one or more stock weights.
    """

    chromosome = chromosome.copy()

    if np.random.random() < mutation_rate:

        # Select a random gene
        gene = np.random.randint(number_of_stocks)

        # Add a small random change
        chromosome[gene] += np.random.normal(0, 0.05)

        # Prevent negative weights
        chromosome[gene] = max(chromosome[gene], 0)

        # Normalize weights
        if np.sum(chromosome) > 0:
            chromosome = chromosome / np.sum(chromosome)

    return chromosome


# ------------------------------------------------------------
# 8. GENETIC ALGORITHM ITERATION
# ------------------------------------------------------------

population = create_population()

best_solution = None
best_fitness = -np.inf

fitness_history = []

for generation in range(number_of_generations):

    # Evaluate fitness of every chromosome
    fitness_values = np.array([
        fitness_function(chromosome)
        for chromosome in population
    ])

    # Find best chromosome in current generation
    best_index = np.argmax(fitness_values)

    current_best = population[best_index]
    current_best_fitness = fitness_values[best_index]

    # Store global best solution
    if current_best_fitness > best_fitness:

        best_fitness = current_best_fitness
        best_solution = current_best.copy()

    fitness_history.append(best_fitness)

    # Create new population
    new_population = []

    while len(new_population) < population_size:

        # Selection
        parent1 = roulette_wheel_selection(
            population,
            fitness_values
        )

        parent2 = roulette_wheel_selection(
            population,
            fitness_values
        )

        # Crossover
        child1, child2 = arithmetic_crossover(
            parent1,
            parent2
        )

        # Mutation
        child1 = mutation(child1)
        child2 = mutation(child2)

        new_population.append(child1)

        if len(new_population) < population_size:
            new_population.append(child2)

    # Replace old population
    population = np.array(new_population)


# ------------------------------------------------------------
# 9. OUTPUT THE BEST SOLUTION
# ------------------------------------------------------------

print("\n========================================")
print("OPTIMAL PORTFOLIO")
print("========================================")

for i in range(number_of_stocks):

    print(
        f"{stocks[i]} : "
        f"{best_solution[i] * 100:.2f}%"
    )

print("----------------------------------------")

best_return = calculate_return(best_solution)
best_risk = calculate_risk(best_solution)

print(f"Expected Return : {best_return * 100:.2f}%")
print(f"Portfolio Risk  : {best_risk * 100:.2f}%")
print(f"Fitness Value   : {best_fitness:.4f}")

print("========================================")


OPTIMAL PORTFOLIO
Stock A : 13.49%
Stock B : 20.39%
Stock C : 13.16%
Stock D : 52.95%
----------------------------------------
Expected Return : 9.87%
Portfolio Risk  : 8.94%
Fitness Value   : 1.1044
